# Проект 1.
# Бизнес-вопрос: Окупилось ли подорожание 1 марта 2026 года

С 1 марта 2026 года тариф «Базовый» подорожал с 199 до 249 ₽, «Расширенный» - с 399 до 449 ₽. Нужно понять, привело ли увеличение стоимости тарифа к увеличению прибыли, или отток клиентов съел всю разницу. Отличался ли эффект для двух разных тарифов?

### Гипотезы

**Гипотеза 1.** 

Рост цен на тарифы увеличил отток, база активных пользователей сократилась, а выручка снизилась.

**Гипотеза 2.**

В процентном соотношении, цена "Базового" увеличилась вдвое больше, чем цена "Расширенного" - +25% и +12,5% соответственно. Можно предположить, что отток клиентов "Базового" был больше - они отказались от подписки или перешли на "Расширенный".


---
## 1. Подключение и параметры

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine   

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Сутки и месяцы считаем по UTC — так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

In [ ]:
# Все параметры анализа — в одном месте. Поменяли здесь — пересчиталось всё ниже.

TARIFF = {1: "Базовый", 2: "Расширенный"}          
COLORS = {"Базовый": "#C1272D", "Расширенный": "#9CA3AF"}   

PRICE_CHANGE = "2026-03-01"                       

# Окна сравнения: по три месяца до и после подорожания. Правый конец не включён.
BEFORE = ("2025-12-01", "2026-03-01")             # декабрь, январь, февраль
AFTER = ("2026-03-01", "2026-06-01")              # март, апрель, май

# Длинный ряд для графиков: с сентября 2025 по июль 2026 (месяцы, от первого до последнего).
FIRST_MONTH, LAST_MONTH = "2025-09-01", "2026-07-01"

###  Почему такие окна

Подорожание случилось 01.03.2026, поэтому для анализа беру два временных промежутка по 3 месяца каждый: с 01.12 до 01.03 (конец не включен) и с 01.03 до 01.06. Важно отметить, что периоды не абсолютно равны по продолжительности. Из предудущего ДЗ мы знаем, что в данных за апрель нет двух дней, а сами месяцы разной продолжительности. Поэтому все данные будем сравнивать в расчете на день. 
